# Training-fit audit for the continuous-beta controller

This is a bounded diagnostic, not a new architecture. The audit command uses only the controller's 15 training scenes: it reconstructs the exact 64 sampled blocks per image, evaluates the frozen checkpoint on both sampled and all training blocks, and asks the same 35k-parameter architecture to memorize every block of one high-headroom training image. Calibration, validation, and test are not loaded by the audit.

A fresh Kaggle session has no prior checkpoints, so the preparation cells reproduce A3-M and the frozen beta controller when they are absent. That controller-reproduction command follows its original calibration/validation protocol; only the subsequent fit diagnosis is training-only. No manual `False` to `True` switch is required.


In [ ]:
import json, os, subprocess, zipfile
from pathlib import Path
from IPython.display import FileLink, display

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)


In [ ]:
CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
A3M_ROOT = Path("/kaggle/working/experiments_a3m")
RUN_NAME = "a3m_multiscale_degradation_sidd32_seed42_20ep"
A3M_CHECKPOINT = A3M_ROOT / RUN_NAME / "best.pt"
CONTROLLER_ROOT = Path("/kaggle/working/beta_controller")
CONTROLLER_CHECKPOINT = CONTROLLER_ROOT / "beta_controller.pt"
OUTPUT_ROOT = Path("/kaggle/working/controller_fit_audit")
assert CDD11_ROOT.is_dir() and PRETRAINED_ROOT.is_dir()
gpu_names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip().splitlines()
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), f"Select 2xT4; found {gpu_names}"
print("GPUs:", gpu_names)


In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT), "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/controller_fit_input_audit.json",
], check=True)


In [ ]:
if not A3M_CHECKPOINT.is_file():
    subprocess.run([
        "python", "-m", "hybrid_cot_nafnet.run_ablation",
        "--config", "configs/calibration_a3_multiscale.json",
        "--data-root", str(CDD11_ROOT), "--experiments-root", str(A3M_ROOT),
        "--nproc-per-node", "2", "--runs", RUN_NAME,
    ], check=True)
assert A3M_CHECKPOINT.is_file()
print("Frozen restorer:", A3M_CHECKPOINT)


In [ ]:
if not CONTROLLER_CHECKPOINT.is_file():
    subprocess.run([
        "python", "-m", "hybrid_cot_nafnet.train_beta_controller",
        "--checkpoint", str(A3M_CHECKPOINT), "--data-root", str(CDD11_ROOT),
        "--output-dir", str(CONTROLLER_ROOT), "--git-commit", COMMIT,
        "--block-size", "32", "--fixed-beta", "0.99",
        "--blocks-per-image", "64", "--calibration-scenes", "5",
        "--epochs", "20", "--batch-size", "256",
    ], check=True)
assert CONTROLLER_CHECKPOINT.is_file()
print("Frozen controller:", CONTROLLER_CHECKPOINT)


In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_controller_fit",
    "--checkpoint", str(A3M_CHECKPOINT),
    "--controller", str(CONTROLLER_CHECKPOINT),
    "--data-root", str(CDD11_ROOT),
    "--output-dir", str(OUTPUT_ROOT),
    "--git-commit", COMMIT, "--fixed-beta", "0.99",
    "--blocks-per-image", "64", "--memorization-epochs", "200",
], check=True)


In [ ]:
import pandas as pd
summary = json.loads((OUTPUT_ROOT / "summary.json").read_text())
display(summary["diagnosis"])
display({"exact_sampled_blocks": summary["exact_sampled_blocks"], "all_training_blocks": summary["all_training_blocks"]})
display(summary["memorization"])
display(pd.read_csv(OUTPUT_ROOT / "memorization_log.csv").tail(20))
archive = Path("/kaggle/working/controller_fit_audit_results.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            output_zip.write(path, path.relative_to(OUTPUT_ROOT.parent))
display(FileLink(str(archive)))
